# 9 — Contract Reviewer
> Paste in any commercial contract and get back a prioritised risk register, a list of missing standard protections, and a negotiation checklist — every finding tied to the exact clause it came from.

*Run all cells. Swap the input in the final code cell with your own data.*

In [ ]:
%pip install -q langchain-openai langchain-core pydantic python-dotenv
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI


# --- Data shapes ---

class RiskFinding(BaseModel):
    severity: Literal["critical", "high", "medium", "low"]
    category: Literal[
        "liability", "payment", "ip", "termination",
        "confidentiality", "governance", "compliance", "other"
    ]
    clause_reference: str = Field(
        description="Section or clause number where this risk appears, e.g. 'Section 8.2'"
    )
    issue: str = Field(description="One sentence describing what creates the risk")
    implication: str = Field(description="Business impact if this risk materializes")
    recommended_redline: str = Field(
        description="Specific language change or addition to address this risk"
    )


class MissingProtection(BaseModel):
    protection: str = Field(
        description="Standard protection absent from this contract, e.g. 'limitation of liability cap'"
    )
    why_needed: str = Field(description="Why this protection matters for this contract type")
    suggested_clause: str = Field(description="Draft language to add")


class NegotiationPoint(BaseModel):
    priority: Literal["must_have", "should_have", "nice_to_have"]
    topic: str
    current_position: str = Field(description="What the contract currently says on this point")
    target_position: str = Field(description="What you should push for in negotiation")


class ContractReview(BaseModel):
    contract_type: str = Field(
        description="Type of contract, e.g. 'Professional Services Agreement'"
    )
    counterparty: Optional[str] = Field(
        default=None, description="Name of the other party if identifiable in the document"
    )
    governing_law: Optional[str] = Field(
        default=None, description="Governing law and jurisdiction if stated"
    )
    overall_risk: Literal["high", "medium", "low"]
    executive_summary: str = Field(
        description="2-3 sentence summary of the contract's key risks for a senior executive"
    )
    risk_findings: List[RiskFinding] = Field(
        description="All identified risk clauses sorted by severity, each citing its clause reference"
    )
    missing_protections: List[MissingProtection] = Field(
        description="Standard protections that are absent from this contract"
    )
    negotiation_points: List[NegotiationPoint] = Field(
        description="Prioritized list of points to raise in negotiation"
    )


# --- Reviewer logic ---

REVIEWER_SYSTEM = SystemMessage(
    """You are a senior commercial lawyer reviewing a contract on behalf of a client.

Your analysis must cover three areas:

1. RISK FINDINGS — every clause that creates risk for the client
   - You MUST include a clause_reference (section/clause number) for every finding
   - If you cannot point to a specific clause, do not include that finding
   - Quote or closely paraphrase the language that creates the risk
   - recommended_redline must be concrete proposed language, not vague advice like "negotiate this"

2. MISSING PROTECTIONS — standard clauses absent from this contract
   - Only flag protections that are genuinely standard for this contract type
   - Provide draft suggested_clause language the client can propose

3. NEGOTIATION POINTS — prioritised list of what to push for
   - must_have: deal-breakers; walk away if not addressed
   - should_have: important but not fatal
   - nice_to_have: improvements worth raising if the counterparty is cooperative

Be thorough but precise. A finding without a clause reference is worthless."""
)


def run(contract_text: str) -> ContractReview:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    reviewer = REVIEWER_SYSTEM | llm.with_structured_output(ContractReview)
    return reviewer.invoke(f"CONTRACT TO REVIEW:\n\n{contract_text}")

## The scenario

We're reviewing a SaaS reseller agreement where the platform vendor drafted every clause in their favour — including a unilateral right to change pricing at any time, a perpetual data licence that survives termination, and an auto-renewal provision with a 90-day cancellation window. The agent will flag every clause that creates commercial exposure, identify the protections that are missing entirely, and tell you exactly where to push in the negotiation.

In [ ]:
SAAS_RESELLER_AGREEMENT = """
SAAS RESELLER AGREEMENT

This SaaS Reseller Agreement ("Agreement") is entered into as of March 1, 2025,
between CloudPlatform Inc. ("Vendor") and GrowthCo Ltd. ("Reseller").

Section 1. Appointment and Scope
Vendor appoints Reseller as a non-exclusive reseller of the Platform in the territory.
Vendor reserves the right to appoint additional resellers or sell directly to any customer
at its sole discretion without compensation to Reseller.

Section 2. Pricing and Fees
Vendor may adjust the wholesale pricing for the Platform at any time with 7 days written
notice to Reseller. Reseller is responsible for all fees accrued under customer accounts
it manages, regardless of whether those customers pay Reseller.

Section 3. Data
Reseller grants Vendor a perpetual, irrevocable, royalty-free licence to use, reproduce,
and analyse all customer data processed through the Platform for Vendor's product
improvement purposes. This licence survives termination of this Agreement.

Section 4. Term and Renewal
This Agreement has an initial term of 12 months and will automatically renew for successive
12-month periods unless Reseller provides written notice of non-renewal at least 90 days
prior to the end of the then-current term. Vendor may terminate this Agreement immediately
for convenience at any time.

Section 5. Liability
Vendor's total liability to Reseller under this Agreement shall not exceed $500. Reseller
shall indemnify and hold Vendor harmless from any claims arising out of Reseller's
activities, including claims by Reseller's customers, without limitation.

Section 6. Exclusivity and Non-Compete
During the term and for 24 months following termination, Reseller shall not resell,
distribute, or promote any product that competes with the Platform, as determined
by Vendor in its sole discretion.

Section 7. Amendments
Vendor may amend the terms of this Agreement at any time by posting updated terms to
its website. Continued use of the Platform by Reseller constitutes acceptance of the
amended terms.

Section 8. Governing Law
This Agreement is governed by the laws of the State of California, and Reseller
consents to exclusive jurisdiction in San Francisco County.
"""

review = run(SAAS_RESELLER_AGREEMENT)

SEV_ICON = {"critical": "[CRIT]", "high": "[HIGH]", "medium": "[ MED]", "low": "[ LOW]"}
PRI_ICON = {"must_have": "[!!!]", "should_have": "[!! ]", "nice_to_have": "[!  ]"}

print(f"{'='*65}")
print(f"CONTRACT REVIEW — {review.contract_type}")
print(f"{'='*65}")
print(f"Counterparty:  {review.counterparty or 'not identified'}")
print(f"Governing law: {review.governing_law or 'not stated'}")
print(f"Overall risk:  {review.overall_risk.upper()}")
print(f"\nEXECUTIVE SUMMARY")
print(review.executive_summary)

print(f"\nRISK FINDINGS ({len(review.risk_findings)})")
for f in review.risk_findings:
    print(f"\n  {SEV_ICON[f.severity]} [{f.clause_reference}] {f.issue}")
    print(f"     Impact:   {f.implication}")
    print(f"     Redline:  {f.recommended_redline}")

print(f"\nMISSING PROTECTIONS ({len(review.missing_protections)})")
for p in review.missing_protections:
    print(f"\n  - {p.protection}")
    print(f"    Why:  {p.why_needed}")
    print(f"    Add:  {p.suggested_clause}")

print(f"\nNEGOTIATION CHECKLIST ({len(review.negotiation_points)})")
for n in review.negotiation_points:
    print(f"\n  {PRI_ICON[n.priority]} {n.topic}")
    print(f"    Current: {n.current_position}")
    print(f"    Target:  {n.target_position}")

## Use your own data

Replace `SAAS_RESELLER_AGREEMENT` in the cell above with:
- Your contract text (copy-paste the full document as a string)
- Any contract type works: services agreements, NDAs, vendor deals, partnership agreements, employment contracts

The agent will return risk findings tied to the exact clause numbers in your document, the standard protections your contract is missing, and a ranked list of what to push for before signing.